In [0]:
import subprocess

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


command_result_schema = ArrayType(
    StructType([
        StructField("command", StringType(), nullable=False),
        StructField("exit_code", IntegerType(), nullable=True),
        StructField("output", StringType(), nullable=True),
    ])
)


@F.udf(returnType=command_result_schema)
def run_commands_udf(commands: list[str]):
    results = []

    for command in commands:
        try:
            process = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=10,
            )

            output = (
                process.stdout.strip()
                or process.stderr.strip()
                or ""
            )

            results.append({
                "command": command,
                "exit_code": process.returncode,
                "output": output,
            })

        except Exception as exc:
            results.append({
                "command": command,
                "exit_code": None,
                "output": f"{type(exc).__name__}: {exc}",
            })

    return results

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [0]:
def run_worker_commands(commands: list[str]):
    commands_column = F.array(
        *[F.lit(command) for command in commands]
    )

    return (
        spark.range(1)
        .select(
            F.explode(
                run_commands_udf(commands_column)
            ).alias("result")
        )
        .select(
            F.col("result.command").alias("command"),
            F.col("result.exit_code").alias("exit_code"),
            F.col("result.output").alias("output"),
        )
    )

 - Spark Connect

In [0]:
%%bash 
echo "Spark Connect IP"
ip -4 -o addr show dev eth0 | awk '{print $4}' | cut -d/ -f1

Spark Connect IP
192.168.210.33


In [0]:
commands = [
    "hostname",
    "ifconfig",
    "ssh -V",
    "python --version",
]

result_df = run_worker_commands(commands)

display(result_df)

command,exit_code,output
hostname,0,
ifconfig,0,eth0: flags=65 mtu 8881 inet 192.168.210.41 netmask 255.255.255.255 inet6 fddb:face:1::c0a8:d229 prefixlen 128 scopeid 0x0 inet6 fe80::d079:16ff:fe3c:af0b prefixlen 64 scopeid 0x0 ether d2:79:16:3c:af:0b txqueuelen 8881 (Ethernet) RX packets 32 bytes 394975 (394.9 KB) RX errors 0 dropped 0 overruns 0 frame 0 TX packets 18 bytes 1276 (1.2 KB) TX errors 0 dropped 0 overruns 0 carrier 0 collisions 0 device memory 0xbaf3c16000022b1-0 lo: flags=73 mtu 65536 inet 127.0.0.1 netmask 255.0.0.0 inet6 ::1 prefixlen 128 scopeid 0x0 loop txqueuelen 65536 (Local Loopback) RX packets 0 bytes 0 (0.0 B) RX errors 0 dropped 0 overruns 0 frame 0 TX packets 0 bytes 0 (0.0 B) TX errors 0 dropped 0 overruns 0 carrier 0 collisions 0 device memory 0x10000-0
ssh -V,0,"OpenSSH_9.6p1 Ubuntu-3ubuntu13.16, OpenSSL 3.0.13 30 Jan 2024"
python --version,0,Python 3.12.3
